In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import traceback
from matplotlib.animation import FFMpegWriter, PillowWriter
from scipy import signal

BASE_DIR = './Data/some2'

GIF_DIR = os.path.join(BASE_DIR, 'gifs_no_peaks2')

# Sliding Window Parameters
WINDOW_SIZE = 1000  # Number of data points in the window (e.g., 1 second if data is at 10kHz)
STEP_SIZE = 300     # Number of data points to slide the window by (controls overlap and speed)

DATA_COLUMN = '0'
TIME_COLUMN = 'Time'

# Animation Parameters
FPS = 1             # Frames per second for the output GIF
INTERVAL = 500       # Delay between frames in milliseconds (1000 / FPS)

os.makedirs(GIF_DIR, exist_ok=True)
print(f"Base directory: {os.path.abspath(BASE_DIR)}")
print(f"GIF output directory: {os.path.abspath(GIF_DIR)}")

def clamp(value, lower=0.01, upper=0.1):
    return max(lower, min(value, upper))

def get_peaks(acceleration0):
    """
    acceleration0: accelerometer data, measured to the 10,000th of a second
    Returns: x and y coordinates of peaks
    """
    percentile = np.percentile(acceleration0, 99)
    percent_of_max = 0.1 * np.max(acceleration0)
    height = clamp(max(percentile, percent_of_max))
    distance = 350 + 5 / height

    x, y = signal.find_peaks(acceleration0, distance=distance, height=height)
    x = x / 10000  # Convert to seconds
    y = y["peak_heights"]
    return x, y


Base directory: /Users/krishnanshugupta/Cal Poly/Acoustic-Space-Boiling/Data/some2
GIF output directory: /Users/krishnanshugupta/Cal Poly/Acoustic-Space-Boiling/Data/some2/gifs_no_peaks2


In [2]:
def create_gif_sliding_window(csv_file, output_gif):
    fig = None
    try:
        # --- Load Data ---
        data = pd.read_csv(csv_file)
        if TIME_COLUMN not in data.columns or DATA_COLUMN not in data.columns:
            print(f"Skipping {os.path.basename(csv_file)}: Missing required columns.")
            return
        data = data.set_index(TIME_COLUMN)

        time = data.index.to_numpy()
        signal_data = data[DATA_COLUMN].to_numpy()

        if len(time) <= WINDOW_SIZE or len(time) != len(signal_data):
            print(f"Skipping {os.path.basename(csv_file)}: Not enough data or mismatch.")
            return

        # --- Setup Plot ---
        fig, ax = plt.subplots(figsize=(12, 7))
        line, = ax.plot([], [], lw=1.5, color='dodgerblue', label='Connected Peaks')
        scatter = ax.scatter([], [], color='crimson', s=30, label='Peaks')

        ax.set_title(f"Sliding Window Peaks\n{os.path.basename(csv_file)}", fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel("Accelerometer Value", fontsize=12)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend()
        fig.tight_layout()

        def update(frame_index):
            start_index = frame_index
            end_index = start_index + WINDOW_SIZE
            if end_index > len(time): end_index = len(time)
            if start_index >= end_index: return line, scatter

            time_window = time[start_index:end_index]
            signal_window = signal_data[start_index:end_index]

            # Update line
            line.set_data(time_window, signal_window)

            # Get peaks
            peak_x, peak_y = get_peaks(signal_window)
            time_offset = time_window[0]
            peak_x += time_offset  # convert from relative to absolute time

            # Update scatter
            scatter.set_offsets(np.c_[peak_x, peak_y])

            # Axis limits
            ax.set_xlim(time_window[0], time_window[-1])
            window_max = max(signal_window.max(), abs(signal_window.min()), 1e-6)
            y_range = window_max * 1.1  # Add some buffer
            y_pad = 0.005 * y_range if y_range > 1e-9 else 0.02
            ax.set_ylim(signal_window.min() - y_pad, signal_window.max() + y_pad)
            ax.set_ylim(-y_range, y_range)

            return line, scatter

        frames = range(0, len(time) - WINDOW_SIZE + 1, STEP_SIZE)
        ani = animation.FuncAnimation(fig, update, frames=frames, blit=True, interval=INTERVAL)

        print(f"Saving animation to: {output_gif}")
        if output_gif.endswith(".gif"):
            ani.save(output_gif, writer=PillowWriter(fps=FPS))
        elif output_gif.endswith(".mp4"):
            writer = FFMpegWriter(fps=FPS)
            ani.save(output_gif, writer=writer)
        else:
            raise ValueError("Unsupported file extension. Use .gif or .mp4")

    except Exception as e:
        print(f"Error processing {csv_file}: {e}")
        traceback.print_exc()
    finally:
        if fig:
            plt.close(fig)

In [3]:
def process_all_files_sliding_window():
    """ Walks through BASE_DIR and creates sliding window GIFs for each CSV file. """
    print(f"Starting processing in: {BASE_DIR}")
    print(f"Outputting GIFs to: {GIF_DIR}")

    file_count = 0
    processed_count = 0
    error_count = 0

    for root, dirs, files in os.walk(BASE_DIR):
        # IMPORTANT: Skip the GIF output directory itself to prevent recursion
        if os.path.abspath(root).startswith(os.path.abspath(GIF_DIR)):
            print(f"  Skipping GIF output directory: {root}")
            continue

        # Also skip any directory named 'gifs' that isn't the target GIF_DIR
        if 'gifs' in dirs and os.path.abspath(os.path.join(root, 'gifs')) != os.path.abspath(GIF_DIR):
             print(f"  Skipping unrelated 'gifs' directory: {os.path.join(root, 'gifs')}")
             dirs[:] = [d for d in dirs if d != 'gifs']


        for file in files:
            if file.lower().endswith('.csv'):
                file_count += 1
                full_path = os.path.join(root, file)

                # Create a relative path from BASE_DIR to maintain structure in GIF_DIR
                try:
                    rel_path_from_base = os.path.relpath(full_path, BASE_DIR)
                except ValueError:
                     print(f"      Skipping {full_path}: Cannot determine relative path from {BASE_DIR}")
                     error_count += 1
                     continue

                gif_filename = os.path.splitext(rel_path_from_base)[0] + '.mp4'
                gif_path = os.path.join(GIF_DIR, gif_filename)

                # Create necessary subdirectories within GIF_DIR
                os.makedirs(os.path.dirname(gif_path), exist_ok=True)

                print(f"\nProcessing [{file_count}]: {rel_path_from_base}")
                try:
                    create_gif_sliding_window(full_path, gif_path)
                    processed_count += 1
                except Exception as e:
                    # Catch any unexpected errors from create_gif itself
                    print(f"  !! Top-Level Error creating GIF for {rel_path_from_base}: {e}")
                    traceback.print_exc()
                    error_count += 1

    # --- Summary ---
    print(f"\n--- Processing Complete ---")
    print(f"Total CSV files found: {file_count}")
    print(f"Successfully processed: {processed_count}")
    # Calculate skipped/failed count accurately
    skipped_or_failed = file_count - processed_count
    print(f"Files skipped or failed: {skipped_or_failed}")

In [4]:
process_all_files_sliding_window()

Starting processing in: ./Data/some2
Outputting GIFs to: ./Data/some2/gifs_no_peaks2

Processing [1]: MATLAB 4-34 PM Tue, Oct 1, 2024 Run5 .csv
Saving animation to: ./Data/some2/gifs_no_peaks2/MATLAB 4-34 PM Tue, Oct 1, 2024 Run5 .mp4
  Skipping GIF output directory: ./Data/some2/gifs_no_peaks2

--- Processing Complete ---
Total CSV files found: 1
Successfully processed: 1
Files skipped or failed: 0
